# Exploratory Analysis

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from olist.data import Olist
data = Olist().get_data()

### 1 - Run an exploratory analysis with [pandas profiling](https://github.com/pandas-profiling/pandas-profiling)

In [4]:
! pip install pandas-profiling

     |████████████████████████████████| 324 kB 4.3 MB/s eta 0:00:01
     |████████████████████████████████| 352 kB 24.3 MB/s eta 0:00:01
     |████████████████████████████████| 679 kB 86.5 MB/s eta 0:00:01
     |████████████████████████████████| 102 kB 76.6 MB/s eta 0:00:01
     |████████████████████████████████| 77 kB 8.3 MB/s  eta 0:00:01
     |████████████████████████████████| 460 kB 118.1 MB/s eta 0:00:01
  Using cached htmlmin-0.1.12-py3-none-any.whl
     |████████████████████████████████| 296 kB 39.9 MB/s eta 0:00:01
     |████████████████████████████████| 6.9 MB 19.6 MB/s eta 0:00:01
     |████████████████████████████████| 2.1 MB 58.2 MB/s eta 0:00:01
     |████████████████████████████████| 4.7 MB 39.9 MB/s eta 0:00:01
You should consider upgrading via the '/home/mijka/.pyenv/versions/3.9.7/envs/oclass4/bin/python3.9 -m pip install --upgrade pip' command.


In [5]:
# create a new "/reports" folder 
!mkdir -p ../../data/reports

In [6]:
import pandas_profiling
datasets_to_profile = ['orders', 'products', 'sellers',
                  'customers', 'order_reviews',
                  'order_items']

/home/mijka/.pyenv/versions/3.9.7/envs/oclass4/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_8134/1790716945.py:1: DeprecationWarning: `import pandas_profiling` is going to be deprecated by April 1st. Please use `import ydata_profiling` instead.
  import pandas_profiling


In [7]:
# YOUR CODE: Create and save one html report per dataset to profile (it takes some time to run!)
for d in datasets_to_profile:
    print('exporting: '+d)
    profile = data[d].profile_report(title='Report for '+d)
    profile.to_file(output_file="../../data/reports/"+d+'.html');

exporting: orders


Export report to file: 100%|█████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 737.14it/s]


exporting: products


Export report to file: 100%|█████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 360.21it/s]


exporting: sellers


...

Export report to file: 100%|█████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 665.13it/s]


exporting: order_reviews


Export report to file: 100%|█████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 355.81it/s]


exporting: order_items


Export report to file: 100%|█████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 730.21it/s]


Take 10 min to read the reports, and feel free to add insights of your choice to your db.lewagon.org schema

### 2 - Create a matching table

Looking at our schema, it would be best to create a central `matching_table` that will join the most important foreign keys together, for later use

❓Create the `matching_table`, a DataFrame with the following columns (below).  
Use outer joins to make sure you don't lose any information at this stage.

In [8]:
columns_matching_table = [
    "order_id",
    "review_id",
    "customer_id",
    "product_id",
    "seller_id",
]

In [9]:
# Select only the columns of interest in the various dataframes of interest, before proceeding to any merge
orders = data['orders'][['customer_id', 'order_id']]
reviews = data['order_reviews'][['order_id', 'review_id']]
items = data['order_items'][['order_id', 'product_id','seller_id']]

In [10]:
# Inspect the cardinality of each DataFrame using pd.DataFrame.shape and pd.Series.nunique()
print('orders:', orders.shape, orders.customer_id.nunique(), 'unique customer_ids, and', orders.order_id.nunique(), 'unique order_ids')
print('review: ', reviews.shape, reviews.order_id.nunique(), 'unique order_ids and', reviews.review_id.nunique(), 'unique reviews' )
print('items: ', items.shape, items.order_id.nunique(), 'unique order_ids,', items.product_id.nunique(), 
      'unique product_ids, and', items.seller_id.nunique(), 'unique seller_ids')

orders: (99441, 2) 99441 unique customer_ids, and 99441 unique order_ids
review:  (99224, 2) 98673 unique order_ids and 98410 unique reviews
items:  (112650, 3) 98666 unique order_ids, 32951 unique product_ids, and 3095 unique seller_ids


In [11]:
# Carefully merge DataFrames
matching_table = orders.merge(reviews, on='order_id', how='outer').merge(items, on='order_id', how='outer')
matching_table

,customer_id,order_id,review_id,product_id,seller_id
0,9ef432eb6251297304e76186b10a928d,e481f51cbdc54678b7cc49136f2d6af7,a54f0611adc9ed256b57ede6b6eb5114,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9
1,b0830fb4747a6c6d20dea0b8c802d7ef,53cdb2fc8bc7dce0b6741e2150273451,8d5266042046a06655c8db133d120ba5,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962
2,41ce2a54c0b03bf3443c3d931a367089,47770eb9100c2d0c44946d9cf07ec65d,e73b67b67587f7644d5bd1a52deb1b01,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2
3,f88197465ea7920adcdbec7375364d82,949d5b44dbf5de918fe9c16f97b45f8a,359d03e676b3c069f62cadba8dd3f6e8,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106
4,8ab97904e6daea8866dbdbc4fb7aad2c,ad21c59c0840e6cb83a9ceb5573f8159,e50934924e227544ba8246aeb3770dd4,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8
...,...,...,...,...,...
114087,1fca14ff2861355f6e5f14306ff977a7,63943bddc261676b46f01ca7ac2f7bd8,29bb71b2760d0f876dfa178a76bc4734,f1d4ce8c6dd66c47bbaa8c6781c2a923,1f9ab4708f3056ede07124aad39a2554
114088,1aa71eb042121263aafbe80c1b562c9c,83c1379a015df1e13d02aae0204711ab,371579771219f6db2d830d50805977bb,b80910977a37536adeddd63663f916ad,d50d79cb34e38265a8649c383dcffd48
114089,b331b74b18dc79bcdf6532d51e1637c1,11c177c8e97725db2631073c19f07b62,8ab6855b9fe9b812cd03a480a25058a1,d1c427060a0f73f6b889a5c7c61f2ac4,a1043bafd471dff536d0c462352beb48
114090,b331b74b18dc79bcdf6532d51e1637c1,11c177c8e97725db2631073c19f07b62,8ab6855b9fe9b812cd03a480a25058a1,d1c427060a0f73f6b889a5c7c61f2ac4,a1043bafd471dff536d0c462352beb48


In [12]:
# Inspect the cardinality and `nunique` of the final DataFrame. It should match (114100, 5)
print(matching_table.shape)
print('unique values: ')
print(matching_table.nunique())

(114092, 5)
unique values: 
customer_id    99441
order_id       99441
review_id      98410
product_id     32951
seller_id       3095
dtype: int64
